# AIML-04: fine-tune YOLO11 on BDD100K (free Kaggle GPU)

1. *Add Input* → search **BDD100K** and add a dataset that contains `images/100k/{train,val}` and the `det_20` labels
   (or the older `bdd100k_labels_images_{train,val}.json`). Fix `BDD_ROOT` below to match its path.
2. Accelerator **GPU T4 x2**, Internet **On**. Secrets: `GITHUB_REPO`, `WANDB_API_KEY`.
3. Run All (about 3-4 h for 40 epochs on a 20k-image subset). Download `aiml04_yolo.tgz` from Output.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
repo = s.get_secret('GITHUB_REPO')
os.environ['WANDB_API_KEY'] = s.get_secret('WANDB_API_KEY')
!git clone --depth 1 {repo} /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q 'ultralytics>=8.3' wandb motmetrics && pip install -q --no-deps -e .
!yolo settings wandb=True

In [ ]:
BDD_ROOT = '/kaggle/input/bdd100k-dataset/bdd100k/bdd100k'   # <- adjust to your input dataset layout
!ls {BDD_ROOT} {BDD_ROOT}/labels
!python -m perception.data.bdd_to_yolo --bdd-root {BDD_ROOT} --out /kaggle/working/bdd_yolo --train-limit 20000 --val-limit 2000
from IPython.display import Image; Image('reports/figures/class_balance.png')

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')  # COCO-pretrained -> transfer to driving classes
model.train(data='/kaggle/working/bdd_yolo/data.yaml', epochs=40, imgsz=640, batch=32, device=[0, 1],
            project='aiml04-perception', name='yolo11s_bdd', cos_lr=True, close_mosaic=5,
            fliplr=0.5, mosaic=1.0, mixup=0.1, patience=10, plots=True)
metrics = YOLO('aiml04-perception/yolo11s_bdd/weights/best.pt').val(data='/kaggle/working/bdd_yolo/data.yaml', imgsz=640)
print('mAP50-95', metrics.box.map, 'mAP50', metrics.box.map50)
print(dict(zip(metrics.names.values(), metrics.box.maps.round(3))))

In [ ]:
!mkdir -p models && cp aiml04-perception/yolo11s_bdd/weights/best.pt models/yolo11s_bdd.pt
!tar czf /kaggle/working/aiml04_yolo.tgz models reports aiml04-perception/yolo11s_bdd/*.png aiml04-perception/yolo11s_bdd/*.csv
!ls -lh /kaggle/working/aiml04_yolo.tgz